# 1.1 — Tokenizadorea esploratzen: euskara vs ingelesa (SOLUZIOA)

- **Denbora**: 15 min
- **Modalitatea**: taldeka
- **Mota**: esplorazioa + eztabaida

## Helburua

Ulertu OpenAI eta beste LLM-en **tokenizadoreek** (BPE algoritmoa) testua nola zatitzen duten, eta zergatik euskarazko hitzek ingelesezkoek baino token gehiago behar dituzten.

## Egoera

OpenAI-ren tokenizadoreak (`tiktoken`, GPT-4 ereduarena) testua ikusten duenean, modu jakin batean banatzen du. Notebook honetan, `tiktoken` liburutegia erabiliz, web tokenizadorea (https://platform.openai.com/tokenizer) **lokalean errepikatuko** dugu, eta hizkuntzaren araberako kostua neurtuko.

## Soluzio-ohar didaktikoak

Ariketa honen helburua ez da kodea idaztea, baizik eta **zenbakiak ikusi eta interpretatzea**. Espero diren emaitzak:

- Euskarazko esaldia: **17-19 token inguru**.
- Ingelesezko esaldia: **9 token inguru**.
- Token-kopurua aldakorra da `tiktoken` bertsioaren arabera; ez du zertan zehazki bat etorri orientazio-balioarekin.

Irakasle gisa, eztabaidan **kostu-aldea** azpimarratu: API prezioek tokeneko kobratzen dute, beraz, euskarazko produkzio-erabilerak (chatbot-ak, laburpen-sistemak) **2-4 aldiz garestiagoak** dira. Hori **justizia digitalaren** auzi bat da.

In [3]:
# Setup — tiktoken liburutegia
# Ez baduzu instalatuta: pip install tiktoken

import tiktoken

# GPT-4 ereduaren tokenizadorea kargatu (cl100k_base kodifikazioa)
encoder = tiktoken.get_encoding("cl100k_base")
print("Tokenizadorea kargatuta:", encoder.name)

Tokenizadorea kargatuta: cl100k_base


In [4]:
# Lagungarri-funtzioa: testu bat tokenizatu eta emaitzak inprimatzen ditu

def aztertu_testua(testua: str, etiketa: str = ""):
    """Testu bat tokenizatu eta emaitzak inprimatu."""
    # encode-k token-ID osoen zerrenda itzultzen du
    token_idak = encoder.encode(testua)
    # ID bakoitza bere katera bihurtu, ikusgai egiteko
    tokenak = [encoder.decode([t]) for t in token_idak]

    print(f"\n--- {etiketa} ---")
    print(f"Testua: {testua!r}")
    print(f"Token-kopurua: {len(token_idak)}")
    print(f"Tokenak: {tokenak}")
    return len(token_idak)

In [5]:
# 1. Eginkizuna: tokenizatu ariketako lehen esaldia (euskaraz)
esaldi_eu = "Adimen artifizialaren ereduak ulertzen ari naiz Lanbide Heziketan."
kop_eu = aztertu_testua(esaldi_eu, "Esaldia euskaraz")

# Espero den emaitza: 17-19 token. 'artifizialaren' eta 'Heziketan' moduko
# hitzak hainbat tokenetan banatuko ditu ('arti', 'fizi', 'alaren', etab.)
# BPE algoritmoak ezagutzen ez dituen euskal hitz-formak zatikatzen ditu.


--- Esaldia euskaraz ---
Testua: 'Adimen artifizialaren ereduak ulertzen ari naiz Lanbide Heziketan.'
Token-kopurua: 25
Tokenak: ['Ad', 'imen', ' art', 'if', 'iz', 'ial', 'aren', ' er', 'edu', 'ak', ' ul', 'ert', 'zen', ' a', 'ri', ' na', 'iz', ' Lan', 'b', 'ide', ' He', 'zik', 'et', 'an', '.']


In [6]:
# 2. Eginkizuna: tokenizatu ariketako bigarren esaldia (ingelesez)
esaldi_en = "Pneumonoultramicroscopicsilicovolcanoconiosis is a long word."
kop_en = aztertu_testua(esaldi_en, "Esaldia ingelesez")

# Espero den emaitza: ~9-10 token. Hitz luze bakar hori
# 4-5 tokenetan banatuko du, gainerako hitzak token bakar bana.
# Ingelesezko hitz arruntek token bakarra dute corpuseko maiztasunagatik.


--- Esaldia ingelesez ---
Testua: 'Pneumonoultramicroscopicsilicovolcanoconiosis is a long word.'
Token-kopurua: 22
Tokenak: ['P', 'ne', 'um', 'on', 'oul', 'tram', 'icro', 'sc', 'op', 'ics', 'il', 'ic', 'ov', 'ol', 'cano', 'con', 'iosis', ' is', ' a', ' long', ' word', '.']


In [7]:
# 3. Eginkizuna: alderatu hizkuntza desberdinetako esaldi 'baliokideak'

pareak = [
    ("Kaixo, mundua!", "Hello, world!"),
    ("Gaur eguzkia atera da.", "Today the sun came out."),
    ("Adimen artifiziala etorkizuna da.", "Artificial intelligence is the future."),
]

print(f"\n{'EUSKARAZ':<40} {'INGELESEZ':<40} {'Aldea':>6}")
print("-" * 90)
for eu, en in pareak:
    n_eu = len(encoder.encode(eu))
    n_en = len(encoder.encode(en))
    # Aldea: zenbat aldiz token gehiago euskaraz
    aldea = n_eu / n_en if n_en else 0
    print(f"{eu!r:<40} {en!r:<40} {n_eu}/{n_en} ({aldea:.1f}x)")

# Tipikoki euskarazko esaldiek 1.5-2.5 aldiz token gehiago dituzte ingelesezkoek baino.


EUSKARAZ                                 INGELESEZ                                 Aldea
------------------------------------------------------------------------------------------
'Kaixo, mundua!'                         'Hello, world!'                          6/4 (1.5x)
'Gaur eguzkia atera da.'                 'Today the sun came out.'                10/6 (1.7x)
'Adimen artifiziala etorkizuna da.'      'Artificial intelligence is the future.' 13/7 (1.9x)


In [8]:
# 4. Eginkizuna: kostu-estimazioa
# GPT-4 input prezioa 2026an: ~0.005 USD / 1.000 token (orientagarria)
# Imajinatu enpresa batek egunero 10.000 mezu prozesatzen dituela

PREZIOA_1000_TOKEN = 0.005   # USD
MEZU_KOPURUA = 10_000

mezu_eu = "Bezeroaren eskaera prozesatu da eta hurrengo egunean iritsiko da pakete osoa."
mezu_en = "The customer order has been processed and the full package will arrive the next day."

tokens_eu = len(encoder.encode(mezu_eu))
tokens_en = len(encoder.encode(mezu_en))

# 10.000 mezu prozesatzeak duen kostua
kostu_eu = (tokens_eu * MEZU_KOPURUA / 1000) * PREZIOA_1000_TOKEN
kostu_en = (tokens_en * MEZU_KOPURUA / 1000) * PREZIOA_1000_TOKEN

print(f"Mezua euskaraz: {tokens_eu} token --> 10.000 mezurako: {kostu_eu:.2f} USD/eguna")
print(f"Mezua ingelesez: {tokens_en} token --> 10.000 mezurako: {kostu_en:.2f} USD/eguna")
print(f"Kostu-aldea urtean (365 egun): {(kostu_eu - kostu_en) * 365:.0f} USD")

# Tipikoki: 100-500 USD urtean diferentzia (skala honetan); produkzio-erabilera
# masiboetan (txat-aplikazio bat milioi bat mezurekin) diferentzia 10.000+ USD izan daiteke.

Mezua euskaraz: 27 token --> 10.000 mezurako: 1.35 USD/eguna
Mezua ingelesez: 16 token --> 10.000 mezurako: 0.80 USD/eguna
Kostu-aldea urtean (365 egun): 201 USD


## Hausnarketa — erantzun-gida

1. **Zergatik banatu ditu euskarazko eta ingelesezko hitzak modu desberdinean?**

   BPE algoritmoa **maiztasunean oinarritzen** da: entrenatzeko corpusan (Internet, batez ere ingelesez) maiz agertzen diren karaktere-bikoteak fusionatzen ditu token bihurtu arte. GPT-4-ren corpusa nagusiki ingelesa zen, beraz `the`, `is`, `processed` bezalako hitz arruntak token bakarrekoak dira. Euskara gutxi agertzen denez, `artifizialaren` bezalako hitzak ezin ditu osorik gorde eta zatitan banatzen ditu.

2. **Zer pisu (eta zer kostu) du euskaraz idazteak ChatGPT-rekin?**

   - Kostu zuzena: **2-4x gehiago** API deietan.
   - Testuinguru-leiho txikiagoa praktikan: mezu batek 100 hitz ditu, euskaraz 250 token, ingelesez 100 token. Eredu bera erabiltzean, dokumentu motzagoak sartzen dira testuinguru-leiho berean.
   - Latentzia handiagoa: token gehiago = denbora gehiago erantzunaren sorrera-fasean.

3. **Zein estrategia erabiliko zenuke kostuak gutxitzeko?**

   - **Prompta ingelesez**, eta erantzunarekin batera **'Erantzun euskaraz'** mugarekin (erantzunak gehienetan tokens gehiago ditu, baina prompt-aren zatian aurreztu).
   - Eredu **eleaniztunago** bat erabili (esate baterako, Claude eta GPT-5 hobeto egokitzen dira hizkuntza minoritarioetara) edo **euskararako espezializatuta** dauden ereduak (Latxa, IXA-IBM, etab.).
   - **Cache lokala** erantzun komunetarako.

4. **Justizia digitala**:

   Galdera politiko-soziala da: hizkuntz minoritarioko hiztunek **kostu osagarria** ordaintzen dute IA-zerbitzu berberetarako. Hori **diskriminazio ezkutua** izan daiteke. Soluzioak:
   - Euskararako entrenatutako tokenizadoreak garatu (HiTZ zentroa UPV-EHU-n).
   - Administrazio publikoak **eredu propioak** finantzatu (Latxa proiektua).
   - Hizkuntz minoritarioei deskontu bereziak eskaintzeko API politikak.